# Lab 20: Ask the Camera (a Vision-Language Model)

**Duration:** 3 hours. **Level:** beginner. **Work mode:** pairs or individual.

| Student name | Station no. | Date |
|---|---|---|
| | | |

## 1. Overview
The detector of Lab 7 can only name the 80 classes it was trained on. A **vision-language model** (VLM) looks at a picture **and** reads a question in normal language, then answers in words. You can ask it anything: *What is on the table? What colour is the cup? Is the door open? Is anyone wearing a hat?* In this lab you run a small VLM on your Jetson, ask questions about the live camera, compare it with the detector of Lab 7, and build a **scene reporter** that writes a sentence in a log every time the picture changes.

**By the end of this lab you can:**
1. Explain how a vision-language model differs from a detector.
2. Send a camera picture and a question to a VLM and read the answer and the speed.
3. Show how the picture size changes the speed and the answer.
4. Ask questions about the live camera without stopping the video.
5. Build a scene reporter that asks only when the scene changes, and logs the sentences.

**How this notebook works.** Run a cell with **Shift + Enter**, from the top. Type your answers in the cells that say *Your answers*. The live video appears under the cell that starts it. Press **Ctrl + S** often.

> **Memory.** The vision model uses about 2 GB and is shared by all the students of the Jetson. Do not run other big models (training, engines) at the same time.
> **Privacy.** The camera sees people. Use it on yourself and classmates who agree. The picture is processed on the Jetson and is not sent to the internet.
> **Be critical.** The model can describe things that are not in the picture, and it counts badly.

## 2. Background

**How a VLM works.** A VLM joins two networks. A **vision encoder** turns the picture into a list of numbers (like the features of Lab 14). A **language model** (Lab 19) reads these numbers together with your question and writes the answer, token by token. The picture is part of the prompt.

| | Detector (Lab 7) | Vision-language model |
|---|---|---|
| Question it can answer | "Which of my 80 classes are here, and where?" | Any question in words |
| Output | Boxes, classes, confidence | Text |
| Speed | 30 to 100 pictures per second | One answer takes seconds |
| Counting and position | Exact | Often wrong |
| New objects | Needs training (Lab 14) | Often knows them already |
| Best for | Fast, exact, repeated work | Open questions, descriptions, rare cases |

They work **together** in real systems: the detector watches all the time and the VLM is asked only when something interesting happens.

**The model of this lab.** `moondream` is a small VLM (about 1.8 billion parameters, about 1.7 GB). It is made for small devices. Bigger models (`llava`, `qwen2.5vl`) answer better but are slower and use more memory.

**Picture size.** The encoder turns the picture into a fixed number of tokens, so a big picture is not much slower, but a **small** picture loses detail (small text, far objects). The cost is mainly the answer.

**Scene change.** To ask only when needed, the program compares the picture with the one at the last question. `cv2.absdiff` gives the difference per pixel; the mean of the difference is a number for "how much changed". When it is above `CHANGE` and at least `MIN_SECONDS` have passed, the program asks again.

**Change one thing at a time.** If you break the notebook, close it and run `import labview; labview.reset(20)` in a cell of a new notebook (File, New, Notebook) for a fresh copy.

---
# Part A: Prepare (30 min)

### Step 1: Ollama and the Vision Model
Ollama was installed in Lab 19. The cell checks it and downloads the vision model if it is not on the Jetson yet (about 1.7 GB, only the first student needs to download). If Ollama is missing, do Part A of Lab 19 first.

In [ ]:
import csv
import glob
import os
import shutil
import textwrap
import threading
import time
from datetime import datetime

import cv2
import numpy as np

import labview

MODEL = "moondream"          # a small vision-language model. Bigger and better: "llava:7b" (needs much memory), "qwen2.5vl:3b"

print("This notebook is saved in:", os.getcwd())
print("ollama:", shutil.which("ollama") or "MISSING (see Part A of Lab 19)")
have = labview.ollama_models()
print("Models on this Jetson:", have)
if not any(name.startswith(MODEL) for name in have):
    print("Downloading", MODEL, "...")
    labview.ollama_pull(MODEL)
print("Ready:", MODEL)

### Step 2: Ask about a Picture File
First use a picture you know: the bus picture from Lab 7 (4 people, a bus). The function `ask_picture` turns the picture into a JPEG (`cv2.imencode`), sends it with the question and returns the answer and the speed. The picture goes in `images=[...]`.

In [ ]:
def ask_picture(frame, question, width=640, **options):
    """Ask the vision model a question about a picture (a numpy image from OpenCV). Returns the dict of labview.ollama_chat."""
    if frame.shape[1] != width:
        frame = cv2.resize(frame, (width, int(frame.shape[0] * width / frame.shape[1])))
    jpeg = cv2.imencode(".jpg", frame, [cv2.IMWRITE_JPEG_QUALITY, 85])[1].tobytes()
    return labview.ollama_chat([{"role": "user", "content": question}], model=MODEL, images=[jpeg],
                               options={"temperature": 0.2, "num_predict": 100, **options})


picture = next((p for p in ("../models/bus.jpg", "../lab8/bus.jpg") if os.path.exists(p)), None)
bus = cv2.imread(picture)
print("Picture:", picture, bus.shape)
for question in ("Describe this picture in one sentence.", "How many people are in the picture?", "What colour is the bus?"):
    result = ask_picture(bus, question)
    print(f"Q: {question}\nA: {result['text']}\n   [first word {result['first_token_s']:.1f} s, {result['tokens_per_s']:.1f} tokens per second, total {result['total_s']:.1f} s]\n")

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Did the descriptions make sense?:
- How many people did the model count? The real number is 4 (the detector of Lab 7 finds them).:
- Time of the first answer and of the next ones. Why is the first one slower?:


---
# Part B: Questions and Picture Size (40 min)

### Step 1: What Can It Answer?
Try different kinds of questions on the bus picture. Write what is right, wrong or made up.

In [ ]:
QUESTIONS = [
    "What is the weather like?",
    "Is anybody wearing sunglasses?",
    "What is written on the bus?",
    "What is the person on the left doing?",
    "Is there a dog in the picture?",
]
for question in QUESTIONS:
    print("Q:", question)
    print("A:", ask_picture(bus, question)["text"], "\n")

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Which answers were right?:
- Which were wrong or made up? Did the model say 'I do not see' when it should?:
- Which kind of question does it answer best (description, colour, reading, counting, yes or no)?:


### Step 2: The Picture Size
The same question with the picture at 3 widths. The cell measures the time and shows the answer, so you can see what small pictures lose.

In [ ]:
question = "What is written on the bus and what colour is it?"
print("Q:", question)
for width in (160, 320, 640, 960):
    result = ask_picture(bus, question, width=width)
    print(f"width {width:>3}: first word {result['first_token_s']:.1f} s, total {result['total_s']:.1f} s | {result['text'][:110]}")

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Times for the four widths:
- At which width did the answer get worse?:
- Which width would you choose for a live camera and why?:


### Step 3: Detector or VLM?
Lab 7's detector counts objects exactly and fast. Fill in the table for the bus picture after you asked the VLM. For the detector you can run `yolo26n.pt` as in Lab 7, or use the number 4 people.

| Question | VLM answer | Detector answer | Which is better? |
|---|---|---|---|
| How many people? | | | |
| Is there a bus? | | | |
| What colour is the bus? | | (it cannot say) | |
| What is written on the bus? | | (it cannot say) | |
| Time per picture | | | |

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Your table (short):
- In your own words: when would you use the detector and when the VLM?:


---
# Part C: Ask the Live Camera (45 min)

### Step 1: The Live Program
The live view shows the camera. Choose a question in the list and press **Ask**. The program sends the **current picture** to the model in a **background thread**, so the video does not stop. The answer appears at the bottom of the picture. Press **Snapshot** to save the current picture as `snapshot_N.jpg`. Press **Stop** when you are done.

`QUESTIONS_LIVE` is the list of questions in the drop-down list. Add your own.

In [ ]:
QUESTIONS_LIVE = ["Describe what you see in one sentence.", "How many people are in the picture?", "What objects are on the table?",
                  "What colour is the biggest object?", "Is anything held in a hand?", "Is it bright or dark here?"]
run = {"frame": None, "busy": False, "asked": "", "answer": "", "info": "", "note": ""}


def wrap(text, width=62, lines=3):
    """Cut a text into at most `lines` lines of `width` characters (for the picture)."""
    rows = textwrap.wrap(text, width) or [""]
    if len(rows) > lines:
        rows = rows[:lines]
        rows[-1] = rows[-1][:width - 3] + "..."
    return rows


def ask_now(state):
    """The Ask button: ask the question about the current picture, in a thread."""
    if run["busy"] or run["frame"] is None:
        return
    frame, question = run["frame"].copy(), state["Question"]
    run.update(busy=True, asked=question, answer="", info="")

    def work():
        try:
            result = ask_picture(frame, question)
            run.update(answer=result["text"], info=f"{result['total_s']:.1f} s, {result['tokens_per_s']:.1f} tokens per second")
        except SystemExit as error:                        # for example: Ollama is not running
            run["answer"] = "Error: " + str(error)[:120]
        finally:
            run["busy"] = False

    threading.Thread(target=work, daemon=True).start()


def snapshot(state):
    if run["frame"] is not None:
        number = len(glob.glob("snapshot_*.jpg")) + 1
        cv2.imwrite(f"snapshot_{number}.jpg", run["frame"])
        run["note"] = f"Saved snapshot_{number}.jpg"


def process(frame, state):
    run["frame"] = frame.copy()
    shown = frame.copy()
    h, w = shown.shape[:2]
    bar = shown[h - 112:h].copy()
    shown[h - 112:h] = (bar * 0.3).astype(np.uint8)                          # a dark bar for the text
    if run["busy"]:
        lines = ["Thinking ..."]
    elif run["answer"]:
        lines = wrap(run["answer"])
    else:
        lines = ["Press Ask"]
    cv2.putText(shown, "Q: " + (run["asked"] or state["Question"])[:60], (8, h - 92), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 255), 1)
    for i, line in enumerate(lines):
        cv2.putText(shown, line, (8, h - 68 + 24 * i), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 1)
    cv2.putText(shown, f"{state['fps']:.0f} FPS   {run['info']}   {run['note']}", (8, 22), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0, 255, 0), 1)
    return shown


live = labview.live(process, controls={"Question": QUESTIONS_LIVE}, actions={"Ask": ask_now, "Snapshot": snapshot})

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Time of one answer:
- Did the video keep running while the model was thinking?:
- Which questions were answered well on the live camera and which badly?:
- What happens if you press Ask twice quickly? (look at the code of `ask_now`):


---
# Part D: The Scene Reporter (35 min)

### Step 1: Ask Only When Something Changes
A VLM takes seconds, so it cannot look at every picture. The **scene reporter** compares each picture with the one at the last question. When the picture **changed enough** and enough time has passed, it asks `REPORT_QUESTION` and writes the answer in `scene_log.csv` with the time. The function `scene_change` is checked with fake pictures first.

In [ ]:
# ---------------- Settings you can change ----------------
REPORT_QUESTION = "Describe what you see in one short sentence."
CHANGE = 8.0            # how much the picture must change (mean difference, 0 to 255)
MIN_SECONDS = 8.0       # the shortest time between two questions
# ----------------------------------------------------------


def scene_change(old, new):
    """How much the picture changed: the mean absolute difference of the two grey pictures, 0 to 255."""
    a = cv2.GaussianBlur(cv2.cvtColor(cv2.resize(old, (160, 120)), cv2.COLOR_BGR2GRAY), (5, 5), 0)
    b = cv2.GaussianBlur(cv2.cvtColor(cv2.resize(new, (160, 120)), cv2.COLOR_BGR2GRAY), (5, 5), 0)
    return float(cv2.absdiff(a, b).mean())


scene = np.full((480, 640, 3), 120, np.uint8)
person = scene.copy()
cv2.rectangle(person, (200, 100), (420, 460), (30, 30, 200), -1)
dark = (scene * 0.5).astype(np.uint8)
print("Same picture:        ", round(scene_change(scene, scene), 1), "(expected 0)")
print("A person comes in:   ", round(scene_change(scene, person), 1))
print("The light goes down: ", round(scene_change(scene, dark), 1))

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- The three numbers:
- Which is above `CHANGE = 8`? Is that what you expect?:
- How could a small noise in the picture trigger a question? What in `scene_change` helps against it?:


In [ ]:
def write_excel():
    """Rewrite scene_log.xlsx from scene_log.csv, so it opens in Excel with a header."""
    try:
        from openpyxl import Workbook
        from openpyxl.styles import Font
    except ImportError:
        return
    book = Workbook()
    sheet = book.active
    sheet.title = "Scenes"
    with open("scene_log.csv", newline="") as f:
        for row in csv.reader(f):
            sheet.append(row)
    for cell in sheet[1]:
        cell.font = Font(bold=True)
    for column, width in zip("ABCD", (20, 10, 90, 12)):
        sheet.column_dimensions[column].width = width
    book.save("scene_log.xlsx")


def log_scene(change, text, seconds):
    first = not os.path.exists("scene_log.csv")
    with open("scene_log.csv", "a", newline="") as f:
        writer = csv.writer(f)
        if first:
            writer.writerow(["time", "change", "description", "seconds_to_answer"])
        writer.writerow([datetime.now().strftime("%Y-%m-%d %H:%M:%S"), round(change, 1), text, round(seconds, 1)])
    write_excel()


rep = {"frame": None, "ref": None, "last": 0.0, "busy": False, "text": "Waiting for a change ...", "count": 0}


def report_work(frame, change):
    try:
        result = ask_picture(frame, REPORT_QUESTION)
        rep["text"] = result["text"]
        rep["count"] += 1
        log_scene(change, result["text"], result["total_s"])
    except SystemExit as error:
        rep["text"] = "Error: " + str(error)[:100]
    finally:
        rep["busy"] = False


def process_report(frame, state):
    shown = frame.copy()
    h = shown.shape[0]
    now = time.time()
    if rep["ref"] is None:
        rep["ref"] = frame.copy()
    change = scene_change(rep["ref"], frame)
    if not rep["busy"] and change >= state["Change"] and now - rep["last"] >= MIN_SECONDS:
        rep.update(busy=True, last=now, ref=frame.copy())
        threading.Thread(target=report_work, args=(frame.copy(), change), daemon=True).start()
    shown[h - 90:h] = (shown[h - 90:h] * 0.3).astype(np.uint8)
    for i, line in enumerate(wrap(rep["text"], lines=3)):
        cv2.putText(shown, line, (8, h - 62 + 24 * i), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 1)
    cv2.putText(shown, f"change {change:4.1f} (limit {state['Change']:.0f})   reports {rep['count']}   {'THINKING' if rep['busy'] else ''}",
                (8, 24), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0, 255, 255), 1)
    return shown

### Step 2: Run the Scene Reporter
Run the cell. Keep still for a while: nothing should be asked. Then walk in, hold up an object, switch the light, and watch the **change** number and the sentence. The reports are saved in `scene_log.csv` and `scene_log.xlsx`. Press **Stop** when you are done.

In [ ]:
rep.update(ref=None, last=0.0, busy=False, text="Waiting for a change ...", count=0)
live = labview.live(process_report, controls={"Change": (2, 40, 1, CHANGE)})

See the log:

In [ ]:
if os.path.exists("scene_log.csv"):
    with open("scene_log.csv") as f:
        print(f.read())
else:
    print("No reports yet.")

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- How many reports were written?:
- Which changes were reported and which were missed?:
- What did the log say? Are the sentences correct?:
- What did you set `Change` and `MIN_SECONDS` to, and why?:


---
# Part E: Make It Your Own, Test and Present (30 min)

### Step 1: Make It Your Own
Add **at least one improvement** from the list, or one of your own. Change the code in the cells above, run that cell again, then run the live cell again.

- Change `REPORT_QUESTION` to a **purpose**: *Is the door open or closed?*, *Is anybody at the desk?*, *Is the bin full?* and log only the answer.
- Write an **alert** when the answer contains a word (for example `yes`, `open`, `person`).
- Let the **detector of Lab 7** decide when to ask: ask the VLM only when the number of people changes.
- Add the **time and the change** to the picture of the log, or save the picture of each report (`cv2.imwrite`).
- Say the sentence aloud with **Lab 18** (text to speech).
- Try `qwen2.5vl:3b` and compare the speed and the answers.

> **Tip.** Change one thing at a time. To keep a copy of a cell before a big change, copy the cell (select it, press **C**, then **V**) and change the copy.

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Improvement you chose:
- Which lines did you change?:
- Did it work the first time? What went wrong?:


### Step 2: Test
Test each situation and write what happened.

| Test | Situation | Expected | What happened |
|---|---|---|---|
| 1 | "How many people?" with 0, 1, 2 and 3 people in view | The correct number | |
| 2 | "What colour is my shirt?" | The right colour | |
| 3 | Hold a printed text close and ask "What does it say?" | Reads it, or not? | |
| 4 | Ask about something that is not in the picture ("Is there a dog?") | "No" | |
| 5 | A dark room | Better or worse? | |
| 6 | The scene reporter: nobody moves for 1 minute | No report | |
| 7 | The scene reporter: someone walks in | A report within `MIN_SECONDS` | |

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Test 1 to 3:
- Test 4 and 5:
- Test 6 and 7:
- What is the best use of a VLM in your opinion, and where would you not trust it?:


### Step 3: Report and Present
Fill in the report. Then give a **3-minute demonstration**: ask the live camera three questions, show the picture size table, and run the scene reporter with the log.

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Model used and the time of one answer:
- What the VLM does well and badly compared with the detector:
- Power mode used (run  !sudo nvpmodel -q  in a cell, after  labview.sudo_login()  ):
- Your improvement:
- One problem you found and how you fixed it:
- One idea to make it better in future:


> **Where next.** In Lab 21 you join everything: the keyword wakes the Jetson, it listens, thinks and **talks back**, and it can look through the camera when you ask it to.

## 4. Check Task
Show your instructor. The instructor initials each line when it is done correctly.

| No. | Task to demonstrate | Initials |
|---|---|---|
| 1 | The vision model answers questions about the bus picture | |
| 2 | The picture size table is complete and discussed | |
| 3 | The comparison table of the detector and the VLM is complete | |
| 4 | The live camera answers a question without stopping the video | |
| 5 | The three numbers of `scene_change` are explained | |
| 6 | The scene reporter asks only when the scene changes and writes the log | |
| 7 | At least one improvement works and you explain the code you changed | |
| 8 | The test table is complete | |
| 9 | The report is complete and the 3-minute presentation is given | |

## 5. Assessment Questions
Answer in one or two sentences.

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- 1. How does a vision-language model differ from a detector? Give one thing each does better.:
- 2. Describe the two parts of a VLM and what each one does.:
- 3. Why can a detector and a VLM work together in one system? Give an example.:
- 4. Why does the live program ask the model in a thread?:
- 5. Why is a small picture a risk for some questions?:
- 6. What does `scene_change` measure? Why is the picture blurred before it is compared?:
- 7. Why does the scene reporter use `MIN_SECONDS`?:
- 8. The model says there are 3 people but there are 4. Which tool would you use to count exactly?:
- 9. Which task suits a VLM best? (a) Counting 200 cars per minute (b) Describing an unusual event in a picture (c) Measuring exact distances (d) Reading a QR code:


## 6. Troubleshooting

| Symptom | Likely cause | What to try |
|---|---|---|
| `Cannot reach Ollama ...` | The service is not running or not installed | Do Part A of Lab 19. In a terminal: `sudo systemctl start ollama` |
| `model ... not found` | The vision model is not downloaded | Run Step 1 of Part A |
| The download is slow | Big file (1.7 GB), slow internet | Wait. Only one student needs to download. |
| An error about memory | Other models use the 8 GB | Stop other live views, Kernel, Shut Down All Kernels, run again |
| The first answer is very slow | The model is loading | Normal, the next answers are faster |
| `Cannot open the camera` | Another program uses the camera | Press Stop in other live cells, shut down other kernels, ask your teacher to check your camera on the teacher page |
| Pressing Ask does nothing | The model is still thinking | Wait until the `Thinking ...` text goes away |
| The answer is wrong or made up | A small model makes mistakes, especially with counting and reading | Ask simpler questions, use a bigger picture, hold the object closer |
| The scene reporter asks all the time | `Change` is too low, or the camera auto-exposure flickers | Raise the `Change` slider. Fix the light. |
| The scene reporter never asks | `Change` is too high or `MIN_SECONDS` too long | Lower the slider. Check the number at the top of the picture. |
| The notebook looks broken | You changed or deleted too much | Close it and run `import labview; labview.reset(20)` in a cell of a new notebook (File, New, Notebook). You get a fresh copy. |

## 7. Further Reading
- Moondream: https://moondream.ai/
- Ollama vision models: https://ollama.com/search?c=vision
- LLaVA: https://llava-vl.github.io/
- OpenCV, absdiff: https://docs.opencv.org/4.x/d2/de8/group__core__array.html